<a href="https://colab.research.google.com/github/Y4TSK0V/MelBandRoformer_UserFriendly-Colab/blob/main/MelBandRoformer_UserFriendly_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎵 Mel-Band Roformer — Разделение аудио на стемы
###[my telegram](https://Y4TSK0V.t.me)
###[my github](https://github.com/Y4TSK0V)


---



Извлечение вокала, инструментала, караоке-версий и обработка аудио с помощью специализированных нейросетевых моделей.

**Перед началом:** убедитесь, что включён GPU → *Среда выполнения → Сменить среду → T4 GPU*

| Шаг | Действие |
|-----|----------|
| 1 | Выбрать модель |
| 2 | Установить зависимости и скачать модель |
| 3 | Загрузить аудиофайлы |
| 4 | Обработать |
| 5 | Прослушать и скачать результаты |

In [ ]:
#@title ▶️ Шаг 1: Выбор модели { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display, clear_output
import os

# ══════════════════════════════════════════════
#  Проверка GPU
# ══════════════════════════════════════════════
try:
    import torch
    if torch.cuda.is_available():
        _gpu = torch.cuda.get_device_name(0)
        _mem = torch.cuda.get_device_properties(0).total_memory / 1e9
        print(f"✅ GPU: {_gpu} ({_mem:.1f} GB)")
    else:
        print("⚠️  GPU НЕ ОБНАРУЖЕН! Среда выполнения → Сменить среду → GPU")
except ImportError:
    print("⚠️  PyTorch ещё не установлен (будет установлен на шаге 2)")

# ══════════════════════════════════════════════
#  Профили конфигов (генерируются, не скачиваются)
# ══════════════════════════════════════════════

CONFIG_PROFILES = {
    "std": {
        "dim": 384, "depth": 6,
        "stft_normalized": False,
        "chunk_size": 352800, "num_overlap": 2,
        "extra_model": "",
        "label": "Стандартный (913 MB модели)",
    },
    "v7": {
        "dim": 256, "depth": 12,
        "stft_normalized": False,
        "chunk_size": 352800, "num_overlap": 2,
        "extra_model": "",
        "label": "Компактный v7 (490 MB, dim=256 depth=12)",
    },
    "v10": {
        "dim": 256, "depth": 12,
        "stft_normalized": True,
        "chunk_size": 352800, "num_overlap": 2,
        "extra_model": "",
        "label": "Компактный v10 (490 MB, stft_normalized)",
    },
    "karaoke": {
        "dim": 384, "depth": 6,
        "stft_normalized": False,
        "chunk_size": 485100, "num_overlap": 8,
        "extra_model": "",
        "label": "Караоке (913 MB, высокий overlap)",
    },
    "small": {
        "dim": 384, "depth": 6,
        "stft_normalized": False,
        "chunk_size": 352800, "num_overlap": 2,
        "extra_model": "  mlp_expansion_factor: 1\n",
        "label": "Малая (203 MB, mlp_expansion=1) ⚠️",
    },
}


def generate_config(profile_name):
    """Генерирует YAML-конфиг в формате, совместимом с inference.py."""
    p = CONFIG_PROFILES[profile_name]
    extra = p["extra_model"]

    return f"""model:
  dim: {p['dim']}
  depth: {p['depth']}
  stereo: true
  num_stems: 1
  time_transformer_depth: 1
  freq_transformer_depth: 1
  num_bands: 60
  dim_head: 64
  heads: 8
  attn_dropout: 0
  ff_dropout: 0
  flash_attn: True
  dim_freqs_in: 1025
  sample_rate: 44100
  stft_n_fft: 2048
  stft_hop_length: 441
  stft_win_length: 2048
  stft_normalized: {p['stft_normalized']}
  mask_estimator_depth: 2
  multi_stft_resolution_loss_weight: 1.0
  multi_stft_resolutions_window_sizes: !!python/tuple
  - 4096
  - 2048
  - 1024
  - 512
  - 256
  multi_stft_hop_size: 147
  multi_stft_normalized: False
{extra}
training:
  instruments:
  - vocals
  - other
  target_instrument: vocals

inference:
  num_overlap: {p['num_overlap']}
  chunk_size: {p['chunk_size']}
"""


# ══════════════════════════════════════════════
#  Каталог моделей
# ══════════════════════════════════════════════
GABOX = "https://huggingface.co/GaboxR67/MelBandRoformers/resolve/main"
KIMB  = "https://huggingface.co/KimberleyJSN/melbandroformer/resolve/main"

_C = {}  # {категория: [(name, url, profile, size_mb, desc), ...]}

def _a(cat, name, url, profile, sz, desc):
    _C.setdefault(cat, []).append((name, url, profile, sz, desc))

# ── Пути ──
_INS = f"{GABOX}/melbandroformers/instrumental"
_VOC = f"{GABOX}/melbandroformers/vocals"
_KAR = f"{GABOX}/melbandroformers/karaoke"
_EXP = f"{GABOX}/melbandroformers/experimental"

# ═══════════════════════════════════════════════════════
#  ИНСТРУМЕНТАЛЬНЫЕ — стабильные (913 MB, профиль std)
# ═══════════════════════════════════════════════════════
_a("🎸 Инструментальные (стабильные)",
   "Inst GaboxF v9 ⭐ рекомендуется",
   f"{_INS}/Inst_GaboxFv9.ckpt", "std", 913,
   "Лучшее соотношение качества и стабильности.")

_a("🎸 Инструментальные (стабильные)",
   "Inst GaboxF v8",
   f"{_INS}/Inst_GaboxFv8.ckpt", "std", 913,
   "Предыдущая проверенная версия.")

_a("🎸 Инструментальные (стабильные)",
   "Inst GaboxFVX",
   f"{_INS}/Inst_GaboxFVX.ckpt", "std", 913,
   "Версия VX.")

_a("🎸 Инструментальные (стабильные)",
   "Inst Fv4",
   f"{_INS}/inst_Fv4.ckpt", "std", 913,
   "Стабильная, проверенная временем.")

_a("🎸 Инструментальные (стабильные)",
   "Inst GaboxF v7z",
   f"{_INS}/Inst_GaboxFv7z.ckpt", "std", 913,
   "Версия v7z.")

_a("🎸 Инструментальные (стабильные)",
   "INST V7N (шумоподавление)",
   f"{_INS}/INSTV7N.ckpt", "std", 913,
   "V7 со встроенным шумоподавлением.")

_a("🎸 Инструментальные (стабильные)",
   "INST V6",
   f"{_INS}/INSTV6.ckpt", "std", 913,
   "Классическая модель V6.")

_a("🎸 Инструментальные (стабильные)",
   "INST V6N (шумоподавление)",
   f"{_INS}/INSTV6N.ckpt", "std", 913,
   "V6 со встроенным шумоподавлением.")

_a("🎸 Инструментальные (стабильные)",
   "INST V5",
   f"{_INS}/INSTV5.ckpt", "std", 913,
   "Классическая V5.")

_a("🎸 Инструментальные (стабильные)",
   "INST V5N (шумоподавление)",
   f"{_INS}/INSTV5N.ckpt", "std", 913,
   "V5 со встроенным шумоподавлением.")

_a("🎸 Инструментальные (стабильные)",
   "Inst GaboxV7",
   f"{_INS}/Inst_GaboxV7.ckpt", "std", 913,
   "GaBox V7.")

# ═══════════════════════════════════════════════════════
#  ИНСТРУМЕНТАЛЬНЫЕ — компактная (490 MB, профиль v10)
# ═══════════════════════════════════════════════════════
_a("🌸 Инструментальные (компактные)",
   "Inst Flowers V10 ⭐ лёгкая + быстрая",
   f"{_INS}/inst_gaboxFlowersV10.ckpt", "v10", 490,
   "Новая архитектура dim=256 depth=12. Вдвое легче, быстрее.")

# ═══════════════════════════════════════════════════════
#  ИНСТРУМЕНТАЛЬНЫЕ — GaBox classic (913 MB, профиль std)
# ═══════════════════════════════════════════════════════
_a("🎸 Инструментальные (GaBox classic)",
   "inst_gabox (оригинал)",
   f"{_INS}/inst_gabox.ckpt", "std", 913,
   "Первая GaBox инструментальная.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gabox2",
   f"{_INS}/inst_gabox2.ckpt", "std", 913,
   "GaBox v2.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gabox3",
   f"{_INS}/inst_gabox3.ckpt", "std", 913,
   "GaBox v3.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gaboxBv1",
   f"{_INS}/inst_gaboxBv1.ckpt", "std", 913,
   "GaBox B-серия v1.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gaboxBv2",
   f"{_INS}/inst_gaboxBv2.ckpt", "std", 913,
   "GaBox B-серия v2.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gaboxBv3",
   f"{_INS}/inst_gaboxBv3.ckpt", "std", 913,
   "GaBox B-серия v3.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gaboxFv1",
   f"{_INS}/inst_gaboxFv1.ckpt", "std", 913,
   "GaBox F-серия v1.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gaboxFv2",
   f"{_INS}/inst_gaboxFv2.ckpt", "std", 913,
   "GaBox F-серия v2.")

_a("🎸 Инструментальные (GaBox classic)",
   "inst_gaboxFv3",
   f"{_INS}/inst_gaboxFv3.ckpt", "std", 913,
   "GaBox F-серия v3.")

_a("🎸 Инструментальные (GaBox classic)",
   "intrumental_gabox (ранняя)",
   f"{_INS}/intrumental_gabox.ckpt", "std", 913,
   "Ранняя версия (опечатка в имени файла оригинальная).")

# ═══════════════════════════════════════════════════════
#  ВОКАЛЬНЫЕ
# ═══════════════════════════════════════════════════════
_a("🎤 Вокальные",
   "Vocal Fv7 компактная (490 MB)",
   f"{_VOC}/voc_fv7.ckpt", "v7", 490,
   "Новейшая вокальная. dim=256 depth=12. Быстрая и лёгкая.")

_a("🎤 Вокальные",
   "Vocal Fv6",
   f"{_VOC}/voc_fv6.ckpt", "std", 913,
   "Стабильная вокальная v6.")

_a("🎤 Вокальные",
   "Vocal Fv5",
   f"{_VOC}/voc_fv5.ckpt", "std", 913,
   "Вокальная v5.")

_a("🎤 Вокальные",
   "Vocal Fv4",
   f"{_VOC}/voc_fv4.ckpt", "std", 913,
   "Вокальная v4.")

_a("🎤 Вокальные",
   "Vocal Fv3",
   f"{_VOC}/voc_Fv3.ckpt", "std", 913,
   "Вокальная v3.")

_a("🎤 Вокальные",
   "Vocal GaBox Fv2 ⭐",
   f"{_VOC}/voc_gaboxFv2.ckpt", "std", 913,
   "GaBox вокальная Fv2.")

_a("🎤 Вокальные",
   "Vocal GaBox Fv1",
   f"{_VOC}/voc_gaboxFv1.ckpt", "std", 913,
   "GaBox вокальная Fv1.")

_a("🎤 Вокальные",
   "Vocal GaBox 2",
   f"{_VOC}/voc_gabox2.ckpt", "std", 913,
   "GaBox вокальная v2.")

_a("🎤 Вокальные",
   "Vocal GaBox (оригинал)",
   f"{_VOC}/voc_gabox.ckpt", "std", 913,
   "Оригинальная GaBox вокальная.")

# ═══════════════════════════════════════════════════════
#  КАРАОКЕ
# ═══════════════════════════════════════════════════════
_a("🎙️ Караоке",
   "Karaoke GaBox V1",
   f"{_KAR}/Karaoke_GaboxV1.ckpt", "karaoke", 913,
   "Удаляет ведущий вокал. Высокий overlap=8 для качества (медленнее).")

_a("🎙️ Караоке",
   "Small Karaoke (203 MB) ⚠️",
   f"{_KAR}/small_karaoke_gaboxaufr.ckpt", "small", 203,
   "⚠️ Компактная. Может не работать — требует mlp_expansion_factor=1.")

# ═══════════════════════════════════════════════════════
#  ЭКСПЕРИМЕНТАЛЬНЫЕ
# ═══════════════════════════════════════════════════════
_a("🧪 Экспериментальные",
   "Inst Experimental V1",
   f"{_INS}/Inst_ExperimentalV1.ckpt", "std", 913,
   "Экспериментальная инструментальная.")

_a("🧪 Экспериментальные",
   "INST V10",
   f"{_EXP}/INSTV10.ckpt", "std", 913,
   "Инструментальная V10 (экспериментальная).")

_a("🧪 Экспериментальные",
   "INST V9",
   f"{_EXP}/INSTV9.ckpt", "std", 913,
   "Инструментальная V9 (экспериментальная).")

_a("🧪 Экспериментальные",
   "INST V8",
   f"{_EXP}/INSTV8.ckpt", "std", 913,
   "Инструментальная V8 (экспериментальная).")

_a("🧪 Экспериментальные",
   "INST V8N (шумоподавление)",
   f"{_EXP}/INSTV8N.ckpt", "std", 913,
   "V8 со встроенным шумоподавлением.")

_a("🧪 Экспериментальные",
   "Inst V7 Plus",
   f"{_EXP}/instv7plus.ckpt", "std", 913,
   "Улучшенная V7.")

_a("🧪 Экспериментальные",
   "Inst V7 Beta 1",
   f"{_EXP}/instv7beta.ckpt", "std", 913,
   "Бета 1 V7.")

_a("🧪 Экспериментальные",
   "Inst V7 Beta 2",
   f"{_EXP}/instv7beta2.ckpt", "std", 913,
   "Бета 2 V7.")

_a("🧪 Экспериментальные",
   "Inst V7 Beta 3",
   f"{_EXP}/instv7beta3.ckpt", "std", 913,
   "Бета 3 V7.")

_a("🧪 Экспериментальные",
   "Inst Fv9 (exp)",
   f"{_EXP}/Inst_Fv9.ckpt", "std", 913,
   "Экспериментальная Fv9.")

_a("🧪 Экспериментальные",
   "Inst Fv8 (exp)",
   f"{_EXP}/Inst_Fv8.ckpt", "std", 913,
   "Экспериментальная Fv8.")

_a("🧪 Экспериментальные",
   "Inst Fv8b (exp)",
   f"{_EXP}/Inst_FV8b.ckpt", "std", 913,
   "Экспериментальная Fv8b.")

_a("🧪 Экспериментальные",
   "Inst fv7b (exp)",
   f"{_EXP}/inst_fv7b.ckpt", "std", 913,
   "Экспериментальная fv7b.")

_a("🧪 Экспериментальные",
   "Fullness",
   f"{_EXP}/Fullness.ckpt", "std", 913,
   "«Полнота звука» — насыщенный результат.")

_a("🧪 Экспериментальные",
   "Karaoke GaBox V2 (exp)",
   f"{_EXP}/Karaoke_GaboxV2.ckpt", "karaoke", 913,
   "Экспериментальная караоке V2.")

_a("🧪 Экспериментальные",
   "Lead Vocal Dereverb",
   f"{_EXP}/Lead_VocalDereverb.ckpt", "std", 913,
   "Удаление реверберации с вокала.")

_a("🧪 Экспериментальные",
   "Vocal Fv7 Beta 1",
   f"{_EXP}/vocfv7beta1.ckpt", "std", 913,
   "Бета 1 вокальной v7 (913 MB, стандартная архитектура).")

_a("🧪 Экспериментальные",
   "Vocal Fv7 Beta 2",
   f"{_EXP}/vocfv7beta2.ckpt", "std", 913,
   "Бета 2 вокальной v7.")

_a("🧪 Экспериментальные",
   "Vocal Fv7 Beta 3",
   f"{_EXP}/vocfv7beta3.ckpt", "std", 913,
   "Бета 3 вокальной v7.")

_a("🧪 Экспериментальные",
   "kar_gabox (exp караоке)",
   f"{_EXP}/kar_gabox.ckpt", "std", 913,
   "Ранняя экспериментальная караоке.")

_a("🧪 Экспериментальные",
   "BS ResurrectioN (204 MB) ⚠️",
   f"{_EXP}/BS_ResurrectioN.ckpt", "small", 204,
   "⚠️ Малая модель. Может не работать с этим кодом.")

_a("🧪 Экспериментальные",
   "small_inst (203 MB) ⚠️",
   f"{_EXP}/small_inst.ckpt", "small", 203,
   "⚠️ Малая инструментальная. Может не работать с этим кодом.")

# ── Специальные ──
_a("🔧 Специальные",
   "Denoise & Debleed",
   f"{_INS}/denoisedebleed.ckpt", "std", 913,
   "Шумоподавление + удаление перетекания каналов.")

_a("🔧 Специальные",
   "Inst Fv4 Noise",
   f"{_INS}/inst_Fv4Noise.ckpt", "std", 913,
   "Fv4 с шумоподавлением.")

# ── Базовая (KimberleyJSN) ──
_a("📦 Базовая (KimberleyJSN)",
   "MelBandRoformer Original",
   f"{KIMB}/MelBandRoformer.ckpt", "std", 900,
   "Оригинальная модель из исходного репозитория.")


# ══════════════════════════════════════════════
#  Интерфейс
# ══════════════════════════════════════════════
categories = list(_C.keys())

cat_dd = widgets.Dropdown(
    options=categories, value=categories[0],
    description="Категория:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="600px"),
)

def _model_options(cat):
    return [(m[0], i) for i, m in enumerate(_C[cat])]

model_dd = widgets.Dropdown(
    options=_model_options(categories[0]),
    description="Модель:",
    style={"description_width": "100px"},
    layout=widgets.Layout(width="600px"),
)

info_out = widgets.Output()
confirm_out = widgets.Output()

# ── Переключатель: каталог / свой URL ──
mode_toggle = widgets.ToggleButtons(
    options=["📋 Из каталога", "🔗 Свой URL"],
    value="📋 Из каталога",
    style={"button_width": "150px"},
)

custom_url = widgets.Text(
    placeholder="https://huggingface.co/.../resolve/main/model.ckpt",
    description="URL:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="600px", display="none"),
)

custom_profile = widgets.Dropdown(
    options=[(v["label"], k) for k, v in CONFIG_PROFILES.items()],
    value="std",
    description="Профиль:",
    style={"description_width": "70px"},
    layout=widgets.Layout(width="500px", display="none"),
)

def _show_info():
    with info_out:
        clear_output(wait=True)
        if mode_toggle.value == "🔗 Свой URL":
            print("📝 Введите прямую ссылку на .ckpt с HuggingFace")
            print("   и выберите профиль конфига по размеру модели:")
            print()
            for k, v in CONFIG_PROFILES.items():
                print(f"   • {k}: {v['label']}")
            return
        cat = cat_dd.value
        idx = model_dd.value
        name, url, profile, sz, desc = _C[cat][idx]
        prof = CONFIG_PROFILES[profile]
        print(f"📦 {name}")
        print(f"   Размер: ~{sz} MB")
        print(f"   Профиль: {profile} — dim={prof['dim']}, depth={prof['depth']}")
        print(f"   📝 {desc}")
        print(f"   🔗 {url}")

def _on_cat(change):
    model_dd.options = _model_options(change["new"])
    model_dd.value = 0
    _show_info()

def _on_model(change):
    _show_info()

def _on_mode(change):
    is_custom = change["new"] == "🔗 Свой URL"
    for w in [cat_dd, model_dd]:
        w.layout.display = "none" if is_custom else "block"
    for w in [custom_url, custom_profile]:
        w.layout.display = "block" if is_custom else "none"
    _show_info()

cat_dd.observe(_on_cat, names="value")
model_dd.observe(_on_model, names="value")
mode_toggle.observe(_on_mode, names="value")

confirm_btn = widgets.Button(
    description="✔ Подтвердить выбор",
    button_style="success", icon="check",
    layout=widgets.Layout(width="200px"),
)

def _on_confirm(b):
    with confirm_out:
        clear_output(wait=True)
        if mode_toggle.value == "🔗 Свой URL":
            url = custom_url.value.strip()
            if not url or not url.startswith("http"):
                print("❌ Введите корректный URL!")
                return
            profile = custom_profile.value
            sel_name = url.split("/")[-1]
        else:
            cat = cat_dd.value
            idx = model_dd.value
            sel_name, url, profile, sz, desc = _C[cat][idx]

        global SELECTED_MODEL_URL, SELECTED_PROFILE, SELECTED_MODEL_NAME
        SELECTED_MODEL_URL = url
        SELECTED_PROFILE = profile
        SELECTED_MODEL_NAME = sel_name

        prof = CONFIG_PROFILES[profile]
        print(f"✅ Выбрано: {sel_name}")
        print(f"   Профиль: {profile} (dim={prof['dim']}, depth={prof['depth']})")
        print(f"   URL: {url}")
        if "⚠️" in prof["label"]:
            print(f"\n   ⚠️ Модель может не работать с данным inference.py!")
        print(f"\n➡️  Переходите к Шагу 2!")

confirm_btn.on_click(_on_confirm)

print("🎯 Выберите модель для разделения аудио:\n")
display(mode_toggle)
display(cat_dd)
display(model_dd)
display(custom_url)
display(custom_profile)
display(info_out)
display(confirm_btn)
display(confirm_out)
_show_info()

## ⚙️ Шаг 2: Установка и загрузка модели
Запустите ячейку ниже. Установка занимает ~2-3 минуты (зависимости + скачивание модели).

In [ ]:
#@title ▶️ Шаг 2: Установка { display-mode: "form" }

import os, subprocess, shutil, gc
from pathlib import Path
from IPython.display import clear_output

# ── Проверка выбора ──
try:
    _url = SELECTED_MODEL_URL
    _profile = SELECTED_PROFILE
    _name = SELECTED_MODEL_NAME
    _profiles = CONFIG_PROFILES
    _gen_config = generate_config
except NameError:
    raise RuntimeError(
        "❌ Сначала запустите Шаг 1 и подтвердите выбор!\n"
        "   (После перезапуска ядра нужно заново выполнить Шаг 1)"
    )

# ── Пути ──
REPO_DIR    = Path("/content/Mel-Band-Roformer-Vocal-Model")
MODEL_PATH  = REPO_DIR / "model.ckpt"
SONGS_DIR   = Path("/content/songs")
RESULTS_DIR = Path("/content/results")

# ── 1. Клонирование ──
print("📂 Репозиторий...")
if REPO_DIR.exists():
    print("   Уже существует, пропускаем.")
else:
    ret = subprocess.run(
        ["git", "clone", "--depth=1",
         "https://github.com/KimberleyJensen/Mel-Band-Roformer-Vocal-Model",
         str(REPO_DIR)],
        capture_output=True, text=True
    )
    if ret.returncode != 0:
        raise RuntimeError(f"❌ git clone:\n{ret.stderr}")
    print("   ✓ Склонирован.")

# ── 2. Зависимости ──
print("📦 Зависимости...")
for cmd, label in [
    (["pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], "requirements.txt"),
    (["pip", "install", "-q", "librosa", "pydub"], "librosa, pydub"),
]:
    ret = subprocess.run(cmd, capture_output=True, text=True)
    if ret.returncode != 0:
        print(f"   ⚠️ {label}:\n{ret.stderr[:300]}")
    else:
        print(f"   ✓ {label}")

# ── 3. Место на диске ──
stat = shutil.disk_usage("/content")
free_gb = stat.free / 1e9
print(f"💾 Свободно: {free_gb:.1f} GB")
if free_gb < 2.0:
    print("   ⚠️ Мало! Рекомендуется ≥2 GB.")

# ── 4. Генерация конфига ──
prof = _profiles[_profile]
print(f"\n📄 Генерация конфига: профиль «{_profile}»")
print(f"   dim={prof['dim']}, depth={prof['depth']}, "
      f"stft_normalized={prof['stft_normalized']}, "
      f"overlap={prof['num_overlap']}")

config_path = REPO_DIR / "generated_config.yaml"
config_content = _gen_config(_profile)
config_path.write_text(config_content, encoding="utf-8")
print(f"   ✓ Сохранён: {config_path.name}")

if prof.get("extra_model"):
    print(f"   ⚠️ Профиль содержит доп. параметры (mlp_expansion_factor и т.д.)")
    print(f"      Если модель не загрузится — попробуйте другую.")

# ── 5. Скачивание модели ──
print(f"\n🤖 Модель: {_name}")
print(f"   {_url}")

if MODEL_PATH.exists():
    MODEL_PATH.unlink()

ret = subprocess.run(
    ["wget", "--progress=dot:giga", "-O", str(MODEL_PATH), _url],
)
if ret.returncode != 0 or not MODEL_PATH.exists():
    raise RuntimeError("❌ Не удалось скачать модель. Проверьте URL.")

file_size_mb = MODEL_PATH.stat().st_size / 1e6
print(f"   Размер: {file_size_mb:.0f} MB")

# Защита от HTML вместо модели
if file_size_mb < 10:
    with open(MODEL_PATH, "rb") as f:
        head = f.read(200)
    if b"<html" in head.lower() or b"<!doctype" in head.lower():
        MODEL_PATH.unlink()
        raise RuntimeError("❌ Вместо модели скачалась HTML-страница. URL недействителен.")

# Проверка размера vs профиль
expected_ranges = {"std": (800, 1000), "karaoke": (800, 1000),
                   "v7": (400, 600), "v10": (400, 600), "small": (150, 300)}
lo, hi = expected_ranges.get(_profile, (100, 1200))
if not (lo <= file_size_mb <= hi):
    print(f"   ⚠️ Размер {file_size_mb:.0f} MB не соответствует профилю «{_profile}»")
    print(f"      Ожидалось {lo}-{hi} MB. Возможно, неправильный профиль.")

# Проверка PyTorch
import torch
try:
    _test = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
    del _test
    gc.collect()
    print("   ✓ PyTorch-проверка пройдена.")
except Exception as e:
    raise RuntimeError(f"❌ Модель повреждена: {e}")

# ── 6. Рабочие папки ──
SONGS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

# ── 7. Глобальные переменные ──
ACTIVE_CONFIG_PATH = str(config_path)
ACTIVE_MODEL_PATH  = str(MODEL_PATH)

clear_output(wait=True)
print("=" * 55)
print("✅ Установка завершена!")
print(f"   Модель:   {_name}")
print(f"   Профиль:  {_profile} (dim={prof['dim']}, depth={prof['depth']})")
print(f"   Конфиг:   {config_path.name} (сгенерирован)")
print(f"   Размер:   {file_size_mb:.0f} MB")
print(f"   Диск:     {free_gb:.1f} GB свободно")
print("=" * 55)
print("\n📋 Форматы: MP3, WAV, FLAC, M4A, OGG, WMA, AIFF, OPUS")
print("\n⚠️  Для инструментальных моделей: файлы _vocals и")
print("   _instrumental могут быть перепутаны местами.")
print("   Послушайте оба в Шаге 5 и возьмите нужный.")
print("\n➡️  Переходите к Шагу 3!")

## 📤 Шаг 3: Загрузка аудиофайлов
Выберите способ загрузки: с компьютера или из Google Drive.

In [ ]:
#@title ▶️ Шаг 3: Загрузка файлов { display-mode: "form" }

import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
import shutil

SONGS_DIR = Path("/content/songs")
SONGS_DIR.mkdir(exist_ok=True)

AUDIO_EXT = {".mp3", ".wav", ".flac", ".m4a", ".ogg", ".wma",
             ".aiff", ".aif", ".opus", ".webm"}

# ═══════════════════════════════════════
#  Список файлов в очереди
# ═══════════════════════════════════════
file_list_out = widgets.Output()

def refresh_file_list():
    """Показать содержимое папки songs."""
    with file_list_out:
        clear_output(wait=True)
        files = sorted(SONGS_DIR.iterdir()) if SONGS_DIR.exists() else []
        if files:
            total_mb = sum(f.stat().st_size for f in files) / 1e6
            print(f"📂 Файлов в очереди: {len(files)}  ({total_mb:.1f} MB)")
            for f in files:
                sz = f.stat().st_size / 1e6
                print(f"   • {f.name}  ({sz:.1f} MB)")
        else:
            print("📂 Очередь пуста — загрузите аудиофайлы ниже.")

# ═══════════════════════════════════════
#  Кнопка очистки
# ═══════════════════════════════════════
clear_btn = widgets.Button(
    description="🗑️ Очистить очередь",
    button_style="danger",
    icon="trash",
    layout=widgets.Layout(width="220px"),
)
clear_out = widgets.Output()

def on_clear(b):
    with clear_out:
        clear_output(wait=True)
        count = 0
        if SONGS_DIR.exists():
            for f in SONGS_DIR.iterdir():
                f.unlink()
                count += 1
        print(f"🗑️ Удалено: {count}")
    refresh_file_list()

clear_btn.on_click(on_clear)

# ═══════════════════════════════════════
#  Загрузка
# ═══════════════════════════════════════
mode = widgets.ToggleButtons(
    options=["💻 С компьютера", "📁 Google Drive"],
    value="💻 С компьютера",
    style={"button_width": "160px"},
)

gdrive_path = widgets.Text(
    value="/content/drive/MyDrive/music",
    description="Путь:",
    style={"description_width": "50px"},
    layout=widgets.Layout(width="500px", display="none"),
)

def on_mode(change):
    gdrive_path.layout.display = (
        "block" if change["new"] == "📁 Google Drive" else "none"
    )
mode.observe(on_mode, names="value")

upload_btn = widgets.Button(
    description="📥 Загрузить файлы",
    button_style="primary",
    icon="upload",
    layout=widgets.Layout(width="220px"),
)
upload_out = widgets.Output()

def on_upload(b):
    with upload_out:
        clear_output(wait=True)
        added = 0

        if mode.value == "💻 С компьютера":
            from google.colab import files as colab_files
            print("📤 Выберите файлы (можно несколько):\n")
            try:
                uploaded = colab_files.upload()
            except Exception as e:
                print(f"❌ Ошибка загрузки: {e}")
                return
            for fname in uploaded:
                src = Path(fname)
                if src.suffix.lower() not in AUDIO_EXT:
                    print(f"   ⚠️ Пропущен (формат): {fname}")
                    src.unlink(missing_ok=True)
                    continue
                dest = SONGS_DIR / fname
                shutil.move(str(src), str(dest))
                print(f"   ✓ {fname}")
                added += 1

        else:  # Google Drive
            from google.colab import drive
            if not Path("/content/drive").exists():
                print("🔗 Подключение Google Drive...")
                drive.mount("/content/drive")

            src = Path(gdrive_path.value)
            if not src.exists():
                print(f"❌ Не найдено: {src}")
                return

            found = []
            if src.is_file():
                found = [src]
            else:
                found = sorted([p for p in src.iterdir() if p.is_file() and p.suffix.lower() in AUDIO_EXT])

            if not found:
                print(f"❌ Аудиофайлов не найдено в {src}")
                return

            for fp in found:
                dest = SONGS_DIR / fp.name
                shutil.copy2(str(fp), str(dest))
                print(f"   ✓ {fp.name}")
                added += 1

        if added:
            print(f"\n✅ Добавлено: {added}")
            print("➡️  Переходите к Шагу 4!")
        else:
            print("\n⚠️ Ничего не загружено.")

    refresh_file_list()

upload_btn.on_click(on_upload)

# ═══════════════════════════════════════
#  Отображение
# ═══════════════════════════════════════
print("🎵 Управление аудиофайлами\n")
display(file_list_out)
display(widgets.HBox([clear_btn, clear_out]))
display(widgets.HTML("<hr style='margin:10px 0'>"))
display(mode)
display(gdrive_path)
display(upload_btn)
display(upload_out)

refresh_file_list()

## 🚀 Шаг 4: Обработка
Конвертация в WAV (при необходимости) → разделение на стемы.

In [ ]:
#@title ▶️ Шаг 4: Обработка { display-mode: "form" }

import subprocess, time, os
from pathlib import Path

SONGS_DIR   = Path("/content/songs")
RESULTS_DIR = Path("/content/results")
REPO_DIR    = Path("/content/Mel-Band-Roformer-Vocal-Model")

# ── Проверки ──
try:
    _ = ACTIVE_MODEL_PATH, ACTIVE_CONFIG_PATH, SELECTED_MODEL_NAME
except NameError:
    raise RuntimeError("❌ Модель не установлена. Выполните Шаг 2.")

if not SONGS_DIR.exists() or not any(SONGS_DIR.iterdir()):
    raise RuntimeError("❌ Нет файлов. Загрузите аудио в Шаге 3.")

# ── Конвертация не-WAV → WAV через ffmpeg ──
print("🔄 Подготовка файлов...")
converted = 0
for f in list(SONGS_DIR.iterdir()):
    if f.suffix.lower() == ".wav":
        continue
    wav = f.with_suffix(".wav")
    ret = subprocess.run(
        ["ffmpeg", "-y", "-i", str(f), "-ar", "44100", "-ac", "2", str(wav)],
        capture_output=True, text=True
    )
    if ret.returncode == 0 and wav.exists():
        f.unlink()
        converted += 1
    else:
        print(f"   ⚠️ Не удалось конвертировать: {f.name}")
        if ret.stderr:
            print(f"      {ret.stderr.strip()[:200]}")

wav_files = sorted(SONGS_DIR.glob("*.wav"))
if not wav_files:
    raise RuntimeError("❌ Нет WAV-файлов для обработки.")

if converted:
    print(f"   Сконвертировано: {converted}")
print(f"   Готово к обработке: {len(wav_files)} файлов\n")

# ── Очистка результатов ──
RESULTS_DIR.mkdir(exist_ok=True)
for f in RESULTS_DIR.iterdir():
    f.unlink()

# ── Запуск inference ──
cfg_name = Path(ACTIVE_CONFIG_PATH).name
print(f"🎵 Модель: {SELECTED_MODEL_NAME}")
print(f"📄 Конфиг: {cfg_name}")

# Предупреждение для нестандартных конфигов
if "default" not in cfg_name and "config_vocals_mel_band_roformer" not in cfg_name:
    print(f"⚠️  Нестандартный конфиг — если будет ошибка, попробуйте")
    print(f"   модель с конфигом 'default' (913 MB модели).")

print("━" * 55)

start = time.time()

# НЕ перехватываем stdout/stderr — пусть идут прямо в ноутбук.
# Это сохраняет прогресс-бары tqdm и показывает трейсбеки.
ret = subprocess.run(
    ["python", "-u", "inference.py",
     "--config_path", ACTIVE_CONFIG_PATH,
     "--model_path",  ACTIVE_MODEL_PATH,
     "--input_folder", str(SONGS_DIR),
     "--store_dir",    str(RESULTS_DIR)],
    cwd=str(REPO_DIR),
)

elapsed = time.time() - start
print("━" * 55)

results = sorted(RESULTS_DIR.glob("*.wav"))

if ret.returncode != 0 or not results:
    print()
    print("❗" * 25)
    print("❌ ОБРАБОТКА ЗАВЕРШИЛАСЬ С ОШИБКОЙ")
    print("❗" * 25)
    print()
    print("⬆️  Прокрутите ВВЕРХ — там должен быть Python traceback")
    print("   с подробностями ошибки.")
    print()
    print("💡 Частые причины и решения:")
    print("   ┌─────────────────────────────────────────────────────┐")
    print("   │ TypeError / KeyError в конфиге                     │")
    print("   │ → Модель несовместима с конфигом.                   │")
    print("   │   Используйте модель с конфигом 'default' (913 MB) │")
    print("   ├─────────────────────────────────────────────────────┤")
    print("   │ CUDA out of memory                                 │")
    print("   │ → Попробуйте компактную модель (490 / 203 MB)      │")
    print("   ├─────────────────────────────────────────────────────┤")
    print("   │ RuntimeError: shape mismatch                       │")
    print("   │ → Неправильный конфиг для данной модели.            │")
    print("   │   Смените модель или конфиг.                       │")
    print("   └─────────────────────────────────────────────────────┘")
else:
    print(f"\n✅ Готово за {elapsed:.0f} сек ({elapsed/60:.1f} мин)\n")
    print(f"📂 Результаты ({len(results)} файлов):")
    for r in results:
        sz = r.stat().st_size / 1e6
        print(f"   • {r.name}  ({sz:.1f} MB)")
    print("\n➡️  Переходите к Шагу 5!")

## 🎧 Шаг 5: Прослушивание и скачивание
Послушайте результаты прямо в браузере, затем скачайте ZIP-архив.

In [ ]:
#@title ▶️ Шаг 5: Прослушать и скачать { display-mode: "form" }

import subprocess
import IPython.display as ipd
import ipywidgets as widgets
from IPython.display import display, clear_output
from pathlib import Path
from datetime import datetime
import zipfile, re, shutil

RESULTS_DIR = Path("/content/results")
PREVIEW_DIR = Path("/content/_preview")

results = sorted(RESULTS_DIR.glob("*.wav"))

if not results:
    print("❌ Нет результатов. Сначала выполните Шаг 4.")
else:
    # ═══════════════════════════════════════
    #  MP3 превью (лёгкие для браузера)
    # ═══════════════════════════════════════
    PREVIEW_DIR.mkdir(exist_ok=True)
    for f in PREVIEW_DIR.iterdir():
        f.unlink()

    print(f"🎧 Превью результатов ({len(results)} файлов)\n")
    print("   Конвертация в MP3 для лёгкого прослушивания...\n")

    for i, wav in enumerate(results, 1):
        mp3 = PREVIEW_DIR / wav.with_suffix(".mp3").name
        ret = subprocess.run(
            ["ffmpeg", "-y", "-i", str(wav), "-b:a", "128k", "-loglevel", "error", str(mp3)],
            capture_output=True, text=True
        )
        name_lower = wav.name.lower()
        label = "🎤 Вокал" if "vocal" in name_lower else "🎸 Инструментал" if "instrument" in name_lower else "🎵 Стем"
        wav_mb = wav.stat().st_size / 1e6

        if ret.returncode == 0 and mp3.exists():
            display(widgets.HTML(f"<b>[{i}/{len(results)}] {label}:</b> {wav.name} ({wav_mb:.1f} MB)"))
            display(ipd.Audio(filename=str(mp3)))
        else:
            display(widgets.HTML(f"<b>[{i}/{len(results)}] {label}:</b> {wav.name} ({wav_mb:.1f} MB) ⚠️ (WAV)"))
            display(ipd.Audio(filename=str(wav)))
        print()

    for f in PREVIEW_DIR.iterdir():
        f.unlink()

    # ═══════════════════════════════════════
    #  Управление экспортом
    # ═══════════════════════════════════════
    print("━" * 55)

    zip_btn = widgets.Button(description="📦 Скачать ZIP", button_style="success", icon="download", layout=widgets.Layout(width="250px"))
    export_btn = widgets.Button(description="📁 Экспортировать в папку источника", button_style="info", icon="folder-open", layout=widgets.Layout(width="300px"))

    out = widgets.Output()

    def on_zip(b):
        with out:
            clear_output(wait=True)
            try: raw_name = SELECTED_MODEL_NAME
            except: raw_name = "results"
            safe = re.sub(r"[^\w\-]", "_", raw_name).strip("_") or "results"
            zip_path = Path(f"/content/{safe}_{datetime.now().strftime('%H%M%S')}.zip")
            with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
                for r in results: zf.write(r, r.name)
            from google.colab import files as colab_files
            colab_files.download(str(zip_path))
            print(f"✅ ZIP создан и отправлен на скачивание.")

    def on_export(b):
        with out:
            clear_output(wait=True)
            # Пытаемся определить исходный путь из Шага 3 (виджет gdrive_path)
            try:
                import __main__
                source_path = Path(__main__.gdrive_path.value)
                if not source_path.exists():
                    print("❌ Путь источника не найден. Возможно, вы загружали с ПК.")
                    return

                target_dir = source_path if source_path.is_dir() else source_path.parent
                export_path = target_dir / "results"
                export_path.mkdir(exist_ok=True)

                print(f"📤 Экспорт в: {export_path}...")
                for r in results:
                    shutil.copy2(r, export_path / r.name)
                print(f"✅ Успешно скопировано {len(results)} файлов.")
            except Exception as e:
                print(f"❌ Ошибка экспорта: {e}")

    zip_btn.on_click(on_zip)
    export_btn.on_click(on_export)

    display(widgets.HBox([zip_btn, export_btn]))
    display(out)

In [ ]:
#@title 🔄 Быстрая смена модели (без переустановки) { display-mode: "form" }

import subprocess, gc
from pathlib import Path

REPO_DIR   = Path("/content/Mel-Band-Roformer-Vocal-Model")
MODEL_PATH = REPO_DIR / "model.ckpt"

try:
    _url = SELECTED_MODEL_URL
    _profile = SELECTED_PROFILE
    _name = SELECTED_MODEL_NAME
    _profiles = CONFIG_PROFILES
    _gen_config = generate_config
except NameError:
    raise RuntimeError("❌ Сначала выберите модель в Шаге 1!")

if not REPO_DIR.exists():
    raise RuntimeError("❌ Репозиторий не найден. Выполните Шаг 2.")

# ── Генерация конфига ──
config_path = REPO_DIR / "generated_config.yaml"
config_path.write_text(_gen_config(_profile), encoding="utf-8")
prof = _profiles[_profile]
print(f"📄 Конфиг: {_profile} (dim={prof['dim']}, depth={prof['depth']})")

# ── Скачивание модели ──
print(f"🔄 Загрузка: {_name}")
if MODEL_PATH.exists():
    MODEL_PATH.unlink()

subprocess.run(
    ["wget", "--progress=dot:giga", "-O", str(MODEL_PATH), _url],
)

if not MODEL_PATH.exists():
    raise RuntimeError("❌ Не удалось скачать.")

# ── Проверка ──
import torch
try:
    _t = torch.load(MODEL_PATH, map_location="cpu", weights_only=False)
    del _t; gc.collect()
except Exception as e:
    raise RuntimeError(f"❌ Модель повреждена: {e}")

ACTIVE_CONFIG_PATH = str(config_path)
ACTIVE_MODEL_PATH  = str(MODEL_PATH)

file_size_mb = MODEL_PATH.stat().st_size / 1e6
print(f"\n✅ {_name} ({file_size_mb:.0f} MB) | Профиль: {_profile}")
print("➡️  Переходите к Шагу 4!")

📄 Конфиг: std (dim=384, depth=6)
🔄 Загрузка: Vocal Fv6

✅ Vocal Fv6 (913 MB) | Профиль: std
➡️  Переходите к Шагу 4!
